##### FMCG Inventory Data Integration Pipeline - Main Execution Script
##### This notebook orchestrates the ETL pipeline by calling cleaning functions
##### and saving processed data to the processed folder

In [1]:
import pandas as pd
import numpy as np
import logging
import sys
from pathlib import Path
import datetime
import json
import importlib.util
import traceback
from typing import Dict, List, Any
import matplotlib.pyplot as plt

In [2]:
sys.path.append('src')

In [3]:
BASE_DIR = Path.cwd()
print(BASE_DIR)
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
INTEGRATED_DIR = PROCESSED_DIR / 'integrated'
ANALYTICS_DIR = PROCESSED_DIR / 'analytics'
SRC_DIR = BASE_DIR / 'src'
LOG_DIR = BASE_DIR / 'logs'
REPORT_DIR = PROCESSED_DIR / 'validation_reports'

C:\Users\WelCome\Documents\studybuild\Mahshid28


In [4]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
INTEGRATED_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
class AuditLogger:
    """
    Comprehensive audit logging for the ETL pipeline
    
    Tracks:
    - Execution start/end times
    - Files read and their metadata
    - Rows processed at each stage
    - Validation failures
    - Records removed/changed
    - Database load status
    - Errors and exceptions
    """
    
    def __init__(self, log_dir: Path, pipeline_name: str = "FMCG_ETL"):
        """
        Initialize the audit logger
        
        Args:
            log_dir: Directory to store log files
            pipeline_name: Name of the pipeline for identification
        """
        self.log_dir = log_dir
        self.pipeline_name = pipeline_name
        self.execution_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        self.start_time = None
        self.end_time = None
        self.audit_data = {
            'pipeline_name': pipeline_name,
            'execution_id': self.execution_id,
            'start_time': None,
            'end_time': None,
            'duration_seconds': None,
            'status': 'RUNNING',
            'stages': {},
            'files_read': [],
            'rows_processed': {},
            'validation_results': {},
            'cleaning_results': {},
            'integration_results': {},
            'analytics_results': {},
            'errors': [],
            'warnings': []
        }
        
        # Setup file handler for audit log
        self.audit_log_file = log_dir / f'audit_{self.execution_id}.log'
        
        # Create a separate logger for audit
        self.audit_logger = logging.getLogger(f'audit_{self.execution_id}')
        self.audit_logger.setLevel(logging.INFO)
        
        # Remove any existing handlers
        self.audit_logger.handlers.clear()
        
        # File handler
        file_handler = logging.FileHandler(self.audit_log_file, mode='w', encoding='utf-8')
        file_handler.setLevel(logging.INFO)
        file_handler.setFormatter(logging.Formatter(
            '%(asctime)s - %(levelname)s - %(message)s',
            datefmt='%Y-%m-%d %H:%M:%S'
        ))
        self.audit_logger.addHandler(file_handler)
        
        # Also log to console for visibility
        console_handler = logging.StreamHandler(sys.stdout)
        console_handler.setLevel(logging.INFO)
        console_handler.setFormatter(logging.Formatter(
            '%(asctime)s - AUDIT - %(message)s',
            datefmt='%H:%M:%S'
        ))
        self.audit_logger.addHandler(console_handler)
    
    def start(self):
        """Start the audit logging"""
        self.start_time = datetime.datetime.now()
        self.audit_data['start_time'] = self.start_time.isoformat()
        
        self.audit_logger.info("=" * 70)
        self.audit_logger.info(f"AUDIT LOG - {self.pipeline_name}")
        self.audit_logger.info("=" * 70)
        self.audit_logger.info(f"Execution ID: {self.execution_id}")
        self.audit_logger.info(f"Start Time: {self.start_time.strftime('%Y-%m-%d %H:%M:%S')}")
        self.audit_logger.info("=" * 70)
    
    def log_stage_start(self, stage_name: str, description: str = ""):
        """Log the start of a pipeline stage"""
        self.audit_data['stages'][stage_name] = {
            'start_time': datetime.datetime.now().isoformat(),
            'description': description,
            'status': 'STARTED'
        }
        
        self.audit_logger.info("")
        self.audit_logger.info(f">>> STAGE: {stage_name} <<<")
        if description:
            self.audit_logger.info(f"    Description: {description}")
        self.audit_logger.info(f"    Started at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    def log_stage_end(self, stage_name: str, status: str = "SUCCESS", details: dict = None):
        """Log the end of a pipeline stage"""
        if stage_name in self.audit_data['stages']:
            self.audit_data['stages'][stage_name]['end_time'] = datetime.datetime.now().isoformat()
            self.audit_data['stages'][stage_name]['status'] = status
            
            if details:
                self.audit_data['stages'][stage_name]['details'] = details
        
        self.audit_logger.info(f"    Completed: {status}")
        if details:
            for key, value in details.items():
                if isinstance(value, (int, float, str)):
                    self.audit_logger.info(f"      {key}: {value}")
                elif isinstance(value, dict):
                    self.audit_logger.info(f"      {key}:")
                    for k, v in value.items():
                        if isinstance(v, (int, float, str)):
                            self.audit_logger.info(f"        {k}: {v}")
                        else:
                            self.audit_logger.info(f"        {k}: {v}")
                elif isinstance(value, list):
                    if len(value) > 0 and isinstance(value[0], str):
                        self.audit_logger.info(f"      {key}: {len(value)} items")
                    else:
                        self.audit_logger.info(f"      {key}: {value}")
                else:
                    self.audit_logger.info(f"      {key}: {value}")
    
    def log_files_read(self, files: Dict[str, Path]):
        """Log all files that were read"""
        self.audit_data['files_read'] = []
        
        self.audit_logger.info("")
        self.audit_logger.info("--- FILES READ ---")
        
        for name, filepath in files.items():
            if filepath.exists():
                file_size = filepath.stat().st_size
                self.audit_data['files_read'].append({
                    'name': name,
                    'path': str(filepath),
                    'size_bytes': file_size,
                    'size_mb': round(file_size / (1024 * 1024), 2),
                    'modified_time': datetime.datetime.fromtimestamp(filepath.stat().st_mtime).isoformat()
                })
                
                self.audit_logger.info(f"  ✅ {name}: {filepath.name} ({file_size:,} bytes)")
            else:
                self.audit_logger.warning(f"  ❌ {name}: File not found - {filepath}")
    
    def log_rows_processed(self, stage: str, rows_info: Dict[str, int]):
        """Log rows processed at each stage"""
        self.audit_data['rows_processed'][stage] = rows_info
        
        self.audit_logger.info("")
        self.audit_logger.info(f"--- ROWS PROCESSED ({stage}) ---")
        for key, value in rows_info.items():
            self.audit_logger.info(f"  {key}: {value:,}")
    
    def log_validation_results(self, violations: Dict[str, pd.DataFrame]):
        """Log validation failures"""
        self.audit_data['validation_results'] = {}
        
        self.audit_logger.info("")
        self.audit_logger.info("--- VALIDATION RESULTS ---")
        
        total_violations = 0
        for file_name, viol_df in violations.items():
            count = len(viol_df) if viol_df is not None else 0
            self.audit_data['validation_results'][file_name] = {
                'violations': count,
                'violations_by_rule': viol_df['rule_id'].value_counts().to_dict() if not viol_df.empty else {}
            }
            total_violations += count
            
            if count == 0:
                self.audit_logger.info(f"  ✅ {file_name}: No violations")
            else:
                self.audit_logger.warning(f"  ⚠️  {file_name}: {count} violations found")
                if not viol_df.empty:
                    for rule, cnt in viol_df['rule_id'].value_counts().items():
                        self.audit_logger.info(f"      Rule {rule}: {cnt}")
        
        self.audit_data['validation_results']['total_violations'] = total_violations
        if total_violations == 0:
            self.audit_logger.info(f"  ✅ TOTAL: No violations found")
        else:
            self.audit_logger.warning(f"  ⚠️  TOTAL: {total_violations} violations found")
    
    def log_cleaning_results(self, cleaning_summary: Dict[str, Dict]):
        """Log records removed/changed during cleaning"""
        self.audit_data['cleaning_results'] = cleaning_summary
        
        self.audit_logger.info("")
        self.audit_logger.info("--- CLEANING RESULTS ---")
        
        total_initial = 0
        total_final = 0
        total_removed = 0
        
        for file_name, summary in cleaning_summary.items():
            initial = summary.get('initial_rows', 0)
            final = summary.get('final_rows', 0)
            removed = summary.get('removed_rows', 0)
            columns = summary.get('columns', 0)
            
            total_initial += initial
            total_final += final
            total_removed += removed
            
            self.audit_logger.info(f"  {file_name}:")
            self.audit_logger.info(f"    Initial rows: {initial:,}")
            self.audit_logger.info(f"    Final rows: {final:,}")
            self.audit_logger.info(f"    Removed: {removed:,} ({removed/initial*100:.1f}%)" if initial > 0 else "    Removed: 0")
            self.audit_logger.info(f"    Columns: {columns}")
            
            # Log changes made
            if 'changes' in summary:
                self.audit_logger.info(f"    Changes: {summary['changes']}")
        
        self.audit_logger.info("")
        self.audit_logger.info(f"  TOTAL:")
        self.audit_logger.info(f"    Initial rows: {total_initial:,}")
        self.audit_logger.info(f"    Final rows: {total_final:,}")
        self.audit_logger.info(f"    Removed: {total_removed:,}")
    
    def log_integration_results(self, integration_summary: Dict):
        """Log data integration results"""
        self.audit_data['integration_results'] = integration_summary
        
        self.audit_logger.info("")
        self.audit_logger.info("--- INTEGRATION RESULTS ---")
        
        if integration_summary:
            source_files = integration_summary.get('source_files', {})
            for name, count in source_files.items():
                self.audit_logger.info(f"  {name}: {count:,} rows")
            
            self.audit_logger.info(f"  Final integrated: {integration_summary.get('final_rows', 0):,} rows, {integration_summary.get('final_columns', 0)} columns")
            
            # Log join statistics
            join_stats = integration_summary.get('join_statistics', {})
            for join_name, stats in join_stats.items():
                if isinstance(stats, dict):
                    self.audit_logger.info(f"  {join_name}: {stats.get('joined_rows', 'N/A')} rows")
                    if stats.get('unmatched_left_keys', 0) > 0:
                        self.audit_logger.warning(f"    Unmatched in left: {stats['unmatched_left_keys']}")
                    if stats.get('unmatched_right_keys', 0) > 0:
                        self.audit_logger.warning(f"    Unmatched in right: {stats['unmatched_right_keys']}")
    
    def log_analytics_results(self, analytics_summary: Dict):
        """Log analytics (Q5/Q6) results"""
        self.audit_data['analytics_results'] = analytics_summary
        
        self.audit_logger.info("")
        self.audit_logger.info("--- ANALYTICS RESULTS ---")
        
        if analytics_summary:
            for key, value in analytics_summary.items():
                if isinstance(value, (int, float, str)):
                    self.audit_logger.info(f"  {key}: {value}")
                elif isinstance(value, dict):
                    self.audit_logger.info(f"  {key}:")
                    for k, v in value.items():
                        if isinstance(v, (int, float, str)):
                            self.audit_logger.info(f"    {k}: {v}")
                else:
                    self.audit_logger.info(f"  {key}: {value}")
    
    def log_db_load_status(self, db_path: Path, tables: Dict[str, int]):
        """Log database load status"""
        self.audit_data['db_load_status'] = {
            'db_path': str(db_path),
            'exists': db_path.exists(),
            'tables': tables
        }
        
        self.audit_logger.info("")
        self.audit_logger.info("--- DATABASE LOAD STATUS ---")
        
        if db_path.exists():
            self.audit_logger.info(f"  ✅ Database: {db_path.name} ({db_path.stat().st_size:,} bytes)")
        else:
            self.audit_logger.warning(f"  ❌ Database not found: {db_path}")
        
        for table_name, row_count in tables.items():
            self.audit_logger.info(f"    {table_name}: {row_count:,} rows")
    
    def log_error(self, error: Exception, context: str = ""):
        """Log an error"""
        error_info = {
            'timestamp': datetime.datetime.now().isoformat(),
            'context': context,
            'error_type': type(error).__name__,
            'error_message': str(error),
            'traceback': traceback.format_exc()
        }
        self.audit_data['errors'].append(error_info)
        
        self.audit_logger.error("")
        self.audit_logger.error(f"--- ERROR ---")
        if context:
            self.audit_logger.error(f"  Context: {context}")
        self.audit_logger.error(f"  Type: {type(error).__name__}")
        self.audit_logger.error(f"  Message: {str(error)}")
        self.audit_logger.error(f"  Traceback:")
        for line in traceback.format_exc().split('\n')[-5:]:
            if line.strip():
                self.audit_logger.error(f"    {line}")
    
    def log_warning(self, message: str, context: str = ""):
        """Log a warning"""
        warning_info = {
            'timestamp': datetime.datetime.now().isoformat(),
            'context': context,
            'message': message
        }
        self.audit_data['warnings'].append(warning_info)
        
        self.audit_logger.warning(f"  ⚠️  {message}" + (f" ({context})" if context else ""))
    
    def finish(self, status: str = "SUCCESS"):
        """Finish the audit logging"""
        self.end_time = datetime.datetime.now()
        self.audit_data['end_time'] = self.end_time.isoformat()
        self.audit_data['duration_seconds'] = (self.end_time - self.start_time).total_seconds()
        self.audit_data['status'] = status
        
        self.audit_logger.info("")
        self.audit_logger.info("=" * 70)
        self.audit_logger.info("AUDIT LOG - FINISHED")
        self.audit_logger.info("=" * 70)
        self.audit_logger.info(f"Status: {status}")
        self.audit_logger.info(f"End Time: {self.end_time.strftime('%Y-%m-%d %H:%M:%S')}")
        self.audit_logger.info(f"Duration: {self.audit_data['duration_seconds']:.2f} seconds")
        
        if self.audit_data['errors']:
            self.audit_logger.info(f"Errors: {len(self.audit_data['errors'])}")
        if self.audit_data['warnings']:
            self.audit_logger.info(f"Warnings: {len(self.audit_data['warnings'])}")
        
        self.audit_logger.info("=" * 70)
        
        # Save audit data as JSON
        audit_json_path = self.log_dir / f'audit_{self.execution_id}.json'
        with open(audit_json_path, 'w') as f:
            json.dump(self.audit_data, f, indent=2, default=str)
        
        self.audit_logger.info(f"Audit data saved to: {audit_json_path}")
    
    def get_summary(self) -> Dict:
        """Get a summary of the audit data"""
        return {
            'execution_id': self.execution_id,
            'status': self.audit_data['status'],
            'start_time': self.audit_data['start_time'],
            'duration_seconds': self.audit_data['duration_seconds'],
            'files_read': len(self.audit_data['files_read']),
            'stages_completed': len([s for s in self.audit_data['stages'].values() if s.get('status') == 'SUCCESS']),
            'total_errors': len(self.audit_data['errors']),
            'total_warnings': len(self.audit_data['warnings'])
        }

In [6]:
logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

In [7]:
def import_from_notebook(notebook_path):
    """
    Import functions from a Jupyter notebook file
    
    Args:
        notebook_path: Path to the .ipynb file
    
    Returns:
        Module object containing all functions defined in the notebook
    """
    import nbformat
    from nbconvert import PythonExporter
    
    # Read the notebook
    with open(notebook_path, 'r', encoding='utf-8') as f:
        nb = nbformat.read(f, as_version=4)
    
    # Convert to Python code
    exporter = PythonExporter()
    python_code, _ = exporter.from_notebook_node(nb)
    
    # Create a module from the code
    module_name = notebook_path.stem
    spec = importlib.util.spec_from_loader(module_name, loader=None)
    module = importlib.util.module_from_spec(spec)
    
    # Execute the code in the module
    exec(python_code, module.__dict__)
    
    return module

In [8]:
extract_module = import_from_notebook(SRC_DIR / 'extract.ipynb')
load_raw_files = extract_module.load_raw_files
logger.info("Successfully imported functions from extract.ipynb")

In [9]:
transform_module = import_from_notebook(SRC_DIR / 'transform.ipynb')

standardize_column_names = transform_module.standardize_column_names
alias_columns = transform_module.alias_columns
standardize_dates = transform_module.standardize_dates
clean_whitespace = transform_module.clean_whitespace
normalize_case = transform_module.normalize_case
strip_non_alpha = transform_module.strip_non_alpha
coerce_numeric = transform_module.coerce_numeric
standardize_currency = getattr(transform_module, 'standardize_currency', None)
normalize_ids = getattr(transform_module, 'normalize_ids', None)

logger.info("Successfully imported functions from transform.ipynb")

In [10]:
integrate_module = import_from_notebook(SRC_DIR / 'integrate.ipynb')
run_integration_pipeline = integrate_module.run_integration_pipeline
analyze_unmatched_keys = integrate_module.analyze_unmatched_keys
save_integrated_data = integrate_module.save_integrated_data
logger.info("Successfully imported functions from integrate.ipynb")

In [11]:
analytics_module = import_from_notebook(SRC_DIR / 'analytics.ipynb')
run_q5_analytics = analytics_module.run_q5_analytics
run_q6_analysis = analytics_module.run_q6_analysis
logger.info("Successfully imported functions from analytics.ipynb")

In [12]:
validate_module = import_from_notebook(SRC_DIR / 'validate.ipynb')
validate_all_files = validate_module.validate_all_files
generate_validation_report = validate_module.generate_validation_report
logger.info("Successfully imported functions from validate.ipynb")

In [13]:
def clean_sales(df):
    """
    Clean the sales.csv file
    
    Args:
        df: Raw sales dataframe
    
    Returns:
        Cleaned sales dataframe
    """
    logger.info(f"Cleaning sales data - initial rows: {len(df)}")
    
    # Standardize column names
    df = standardize_column_names(df)
    
    # Apply column aliases for consistency
    df = alias_columns(df, {
        'invoice_no': 'transaction_id',
        'stock_code': 'sku',
        'quantity': 'sales_quantity',
        'unit_price': 'price',
        'customer_id': 'customer_id'
    })
    
    # Clean whitespace from all string columns
    text_cols=['sku', 'warehouse_id', 'customer_id']
    df = clean_whitespace(df, text_cols)
    
    # Standardize dates
    date_columns = ['invoice_date', 'transaction_date']
    df = standardize_dates(df, date_columns)
    
    # Coerce numeric columns
    numeric_cols = ['sales_quantity', 'price', 'revenue']
    df = coerce_numeric(df, numeric_cols)
    
    # Clean text columns
    text_cols = ['sku', 'warehouse_id', 'customer_id']
    df = strip_non_alpha(df, text_cols=text_cols)
    df = normalize_case(df, text_cols)
    """
    for col in text_cols:
        if col in df.columns:
            df[col] = normalize_case(df[col], text_cols)
    """
    
    # Remove duplicates based on key columns
    if 'transaction_id' in df.columns and 'sku' in df.columns:
        df = df.drop_duplicates(subset=['transaction_id', 'sku'], keep='first')
    
    logger.info(f"Cleaned sales data - final rows: {len(df)}")
    return df

In [14]:
def clean_orders(df):
    """
    Clean the orders.csv file
    
    Args:
        df: Raw orders dataframe
    
    Returns:
        Cleaned orders dataframe
    """
    logger.info(f"Cleaning orders data - initial rows: {len(df)}")
    
    # Standardize column names
    df = standardize_column_names(df)
    
    # Apply column aliases
    df = alias_columns(df, {
        'order_id': 'order_id',
        'order_date': 'order_date',
        'sku': 'sku',
        'warehouse_id': 'warehouse_id',
        'quantity': 'ordered_quantity',
        'status': 'order_status'
    })
    
    # Clean whitespace
    text_cols=['order_id', 'sku', 'warehouse_id', 'order_status']
    df = clean_whitespace(df, text_cols)
    
    # Standardize dates
    df = standardize_dates(df, ['order_date', 'created_date'])
    
    # Coerce numeric columns
    df = coerce_numeric(df, ['ordered_quantity'])
    
    # Clean text columns
    text_cols = ['sku', 'warehouse_id', 'order_status']
    df = strip_non_alpha(df, text_cols=text_cols)
    df = normalize_case(df, text_cols)
    """
    for col in text_cols:
        if col in df.columns:
            df[col] = normalize_case(df[col], text_cols)
    """
    
    # Remove duplicates
    if 'order_id' in df.columns and 'sku' in df.columns:
        df = df.drop_duplicates(subset=['order_id', 'sku'], keep='first')
    
    logger.info(f"Cleaned orders data - final rows: {len(df)}")
    return df

In [15]:
def clean_inventory(df):
    """
    Clean the inventory.csv file
    
    Args:
        df: Raw inventory dataframe
    
    Returns:
        Cleaned inventory dataframe
    """
    logger.info(f"Cleaning inventory data - initial rows: {len(df)}")
    
    # Standardize column names
    df = standardize_column_names(df)
    
    # Apply column aliases
    df = alias_columns(df, {
        'snapshot_date': 'snapshot_date',
        'sku': 'sku',
        'warehouse_id': 'warehouse_id',
        'on_hand': 'on_hand_quantity',
        'stock': 'on_hand_quantity'
    })
    
    # Clean whitespace
    text_cols=['sku', 'warehouse_id']
    df = clean_whitespace(df, text_cols)
    
    # Standardize dates
    df = standardize_dates(df, ['snapshot_date'])
    
    # Coerce numeric columns
    df = coerce_numeric(df, ['on_hand_quantity'])
    
    # Clean text columns
    df = strip_non_alpha(df, text_cols=text_cols)
    df = normalize_case(df, text_cols)
    """
    for col in text_cols:
        if col in df.columns:
            df[col] = normalize_case(df[col], text_cols)
    """
    
    # Remove duplicates
    if 'sku' in df.columns and 'warehouse_id' in df.columns and 'snapshot_date' in df.columns:
        df = df.drop_duplicates(subset=['sku', 'warehouse_id', 'snapshot_date'], keep='first')
    
    # Remove negative stock (should be flagged in validation)
    if 'on_hand_quantity' in df.columns:
        df = df[df['on_hand_quantity'] >= 0]
    
    logger.info(f"Cleaned inventory data - final rows: {len(df)}")
    return df

In [16]:
def clean_warehouses(df):
    """
    Clean the warehouses.csv file
    
    Args:
        df: Raw warehouses dataframe
    
    Returns:
        Cleaned warehouses dataframe
    """
    logger.info(f"Cleaning warehouses data - initial rows: {len(df)}")
    
    # Standardize column names
    df = standardize_column_names(df)
    
    # Apply column aliases
    df = alias_columns(df, {
        'warehouse_id': 'warehouse_id',
        'name': 'warehouse_name',
        'city': 'city',
        'region': 'region'
    })
    
    # Clean whitespace
    text_cols=['warehouse_id', 'warehouse_name', 'city', 'region']
    df = clean_whitespace(df, text_cols)
    
    df = strip_non_alpha(df, text_cols=text_cols)
    df = normalize_case(df, text_cols)
    """
    for col in text_cols:
        if col in df.columns:
            df[col] = normalize_case(df[col], text_cols)
    """
    
    # Remove duplicates
    if 'warehouse_id' in df.columns:
        df = df.drop_duplicates(subset=['warehouse_id'], keep='first')
    
    logger.info(f"Cleaned warehouses data - final rows: {len(df)}")
    return df

In [17]:
# Verify reproducibility (Q7)
def verify_reproducibility():
    """
    Verify that the pipeline is reproducible (Q7)
    Checks:
    - All required files exist
    - No hard-coded paths (using pathlib)
    - All dependencies are documented
    - Pipeline can be run without manual intervention
    """
    logger.info("\n" + "=" * 60)
    logger.info("Q7: VERIFYING REPRODUCIBILITY")
    logger.info("=" * 60)
    
    checks = []
    
    # 1. Check for README.md
    readme_path = BASE_DIR / 'README.md'
    checks.append(('README.md', readme_path.exists()))
    
    # 2. Check for requirements.txt
    req_path = BASE_DIR / 'requirements.txt'
    checks.append(('requirements.txt', req_path.exists()))
    
    # 3. Check for run_pipeline.py
    run_path = BASE_DIR / 'run_pipeline.py'
    checks.append(('run_pipeline.py', run_path.exists()))
    
    # 4. Check all source notebooks
    src_dir = BASE_DIR / 'src'
    notebooks = ['extract.ipynb', 'transform.ipynb', 'validate.ipynb', 
                 'integrate.ipynb', 'analytics.ipynb']
    for nb in notebooks:
        checks.append((f'src/{nb}', (src_dir / nb).exists()))
    
    # 5. Check for main.ipynb
    checks.append(('main.ipynb', (BASE_DIR / 'main.ipynb').exists()))
    
    # 6. Check data directories
    checks.append(('data/raw/', RAW_DIR.exists()))
    checks.append(('data/processed/', PROCESSED_DIR.exists()))
    
    # 7. Check for no hard-coded paths (using pathlib)
    # This is a simple check - verify that code uses pathlib
    no_hardcoded = True
    
    # Log results
    logger.info("\nReproducibility Checks:")
    for name, status in checks:
        logger.info(f"  {'✅' if status else '❌'} {name}")
    
    # Summary
    all_passed = all(status for _, status in checks)
    
    logger.info("\n" + "-" * 60)
    if all_passed:
        logger.info("✅ All reproducibility checks passed!")
        logger.info("   The pipeline can be reproduced by another reviewer.")
        logger.info("   Instructions:")
        logger.info("   1. Place raw files in data/raw/")
        logger.info("   2. Run: python run_pipeline.py")
        logger.info("   3. Check output in data/processed/")
    else:
        missing = [name for name, status in checks if not status]
        logger.warning(f"⚠️  Missing {len(missing)} items:")
        for item in missing:
            logger.warning(f"   - {item}")
    
    return all_passed

In [18]:
def create_sqlite_database(processed_dir: Path, db_path: Path) -> Dict[str, int]:
    """
    Create SQLite database from cleaned and integrated data with optimization
    
    Args:
        processed_dir: Directory containing cleaned CSV files
        db_path: Path for the SQLite database file
    
    Returns:
        Dictionary with table names and row counts
    """
    import sqlite3
    import pandas as pd
    import numpy as np
    
    # Create database directory
    db_path.parent.mkdir(parents=True, exist_ok=True)
    
    # حذف دیتابیس قبلی اگر وجود دارد
    if db_path.exists():
        db_path.unlink()
        print(f"  🔄 Removed existing database")
    
    # Connect to SQLite database with optimized settings
    conn = sqlite3.connect(str(db_path))
    
    # تنظیمات بهینه برای کاهش حجم
    conn.execute("PRAGMA page_size = 4096")          # صفحه‌های کوچکتر
    conn.execute("PRAGMA cache_size = 10000")        # کش مناسب
    conn.execute("PRAGMA journal_mode = OFF")        # بدون ژورنال (کاهش حجم)
    conn.execute("PRAGMA synchronous = OFF")         # سرعت بیشتر
    conn.execute("PRAGMA temp_store = MEMORY")       # ذخیره موقت در حافظه
    conn.execute("PRAGMA auto_vacuum = FULL")        # فشرده‌سازی خودکار
    
    tables_loaded = {}
    
    # 1. Load cleaned tables with optimization
    cleaned_files = {
        'sales': 'sales_cleaned.csv',
        'orders': 'orders_cleaned.csv',
        'inventory': 'inventory_cleaned.csv',
        'warehouses': 'warehouses_cleaned.csv'
    }
    
    for table_name, filename in cleaned_files.items():
        filepath = processed_dir / filename
        if filepath.exists():
            df = pd.read_csv(filepath)
            
            # بهینه‌سازی نوع داده‌ها برای کاهش حجم
            df = optimize_dtypes(df)
            
            # ذخیره در دیتابیس با ایندکس
            df.to_sql(table_name, conn, if_exists='replace', index=False)
            tables_loaded[table_name] = len(df)
            print(f"  ✅ Loaded {table_name}: {len(df)} rows")
    
    # 2. Load integrated data with optimization
    integrated_path = processed_dir / 'integrated' / 'master_integrated_data.csv'
    if integrated_path.exists():
        df = pd.read_csv(integrated_path)
        
        # بهینه‌سازی نوع داده‌ها
        df = optimize_dtypes(df)
        
        df.to_sql('integrated_data', conn, if_exists='replace', index=False)
        tables_loaded['integrated_data'] = len(df)
        print(f"  ✅ Loaded integrated_data: {len(df)} rows")
    
    # 3. Create indexes for better performance (and small size)
    cursor = conn.cursor()
    
    # ایندکس‌های اصلی
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_sales_sku ON sales(sku)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_sales_warehouse ON sales(warehouse_id)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_orders_sku ON orders(sku)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_orders_warehouse ON orders(warehouse_id)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_inventory_sku ON inventory(sku)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_inventory_warehouse ON inventory(warehouse_id)")
    
    # 4. Create views (بدون ذخیره‌سازی فیزیکی)
    cursor.execute("""
        CREATE VIEW IF NOT EXISTS inventory_view AS
        SELECT 
            sku,
            warehouse_id,
            warehouse_name,
            on_hand_quantity,
            sales_last_30_days,
            orders_last_30_days,
            days_of_inventory,
            out_of_stock,
            overstocked,
            slow_mover
        FROM integrated_data
    """)
    
    cursor.execute("""
        CREATE VIEW IF NOT EXISTS warehouse_summary AS
        SELECT 
            warehouse_id,
            warehouse_name,
            COUNT(DISTINCT sku) as total_skus,
            SUM(on_hand_quantity) as total_inventory,
            SUM(sales_last_30_days) as total_sales_30d,
            SUM(out_of_stock) as out_of_stock_count,
            SUM(overstocked) as overstocked_count,
            SUM(slow_mover) as slow_mover_count
        FROM integrated_data
        GROUP BY warehouse_id, warehouse_name
    """)
    
    # 5. Compact database - VACUUM
    print("  🔄 Compacting database...")
    conn.execute("VACUUM")
    conn.execute("ANALYZE")
    
    conn.close()
    
    # نمایش حجم نهایی
    final_size = db_path.stat().st_size / (1024 * 1024)
    print(f"  ✅ Database created: {final_size:.2f} MB")
    
    return tables_loaded

In [19]:
def optimize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """
    بهینه‌سازی نوع داده‌ها برای کاهش حجم
    """
    df = df.copy()
    
    for col in df.columns:
        # ستون‌های عددی
        if pd.api.types.is_numeric_dtype(df[col]):
            # اگر اعداد صحیح هستند
            if df[col].dropna().apply(lambda x: float(x).is_integer()).all():
                # انتخاب کوچکترین نوع integer
                if df[col].min() >= -128 and df[col].max() <= 127:
                    df[col] = df[col].astype('int8')
                elif df[col].min() >= -32768 and df[col].max() <= 32767:
                    df[col] = df[col].astype('int16')
                elif df[col].min() >= -2147483648 and df[col].max() <= 2147483647:
                    df[col] = df[col].astype('int32')
                else:
                    df[col] = df[col].astype('int64')
            else:
                # اعداد اعشاری
                df[col] = df[col].astype('float32')
        
        # ستون‌های متنی با مقادیر تکراری
        elif pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            # اگر تعداد مقادیر یکتا کم است، از category استفاده کن
            unique_count = df[col].nunique()
            total_count = len(df[col])
            
            if unique_count / total_count < 0.5 and unique_count < 1000:
                try:
                    df[col] = df[col].astype('category')
                except:
                    pass
            
            # محدود کردن طول رشته‌ها
            elif df[col].dtype == 'object':
                max_len = df[col].str.len().max()
                if max_len and max_len < 255:
                    df[col] = df[col].astype('string')
    
    return df

In [20]:
def create_compressed_database(processed_dir: Path, db_path: Path, target_size_mb: int = 20) -> Dict[str, int]:
    """
    ایجاد دیتابیس با حجم هدف مشخص
    """
    import sqlite3
    import pandas as pd
    from pathlib import Path
    
    print(f"\n📊 Creating database with target size: {target_size_mb} MB")
    print("-" * 50)
    
    # ایجاد دیتابیس
    tables = create_sqlite_database(processed_dir, db_path)
    
    # بررسی حجم
    current_size = db_path.stat().st_size / (1024 * 1024)
    
    # اگر حجم بیشتر از حد بود، فشرده‌سازی بیشتر
    if current_size > target_size_mb:
        print(f"\n  ⚠️  Database size ({current_size:.1f} MB) is larger than target ({target_size_mb} MB)")
        print("  🔄 Applying additional compression...")
        
        conn = sqlite3.connect(db_path)
        
        # فشرده‌سازی بیشتر
        conn.execute("PRAGMA page_size = 1024")  # صفحه‌های کوچک‌تر
        conn.execute("VACUUM")
        
        conn.close()
        
        # بررسی مجدد
        new_size = db_path.stat().st_size / (1024 * 1024)
        print(f"  ✅ Compressed to: {new_size:.1f} MB")
        
        if new_size > target_size_mb:
            print(f"  💡 Tip: Consider deleting old data or using more compression")
    
    return tables

In [21]:
def generate_data_quality_summary(
    raw_data: Dict[str, pd.DataFrame],
    processed_data: Dict[str, pd.DataFrame],
    violations: Dict[str, pd.DataFrame],
    summary: Dict
) -> Dict:
    """
    Generate comprehensive data quality summary report
    """
    quality_summary = {
        'generated_at': datetime.datetime.now().isoformat(),
        'overall_quality_score': 0,
        'file_quality': {},
        'data_integrity': {},
        'completeness': {},
        'consistency': {},
        'recommendations': []
    }
    
    total_score = 0
    file_count = 0
    
    for file_name in raw_data.keys():
        raw_df = raw_data.get(file_name)
        clean_df = processed_data.get(file_name)
        
        if raw_df is None or clean_df is None:
            continue
        
        file_quality = {}
        
        # 1. Completeness (missing values)
        missing_before = raw_df.isnull().sum().sum()
        missing_after = clean_df.isnull().sum().sum()
        
        file_quality['completeness'] = {
            'missing_before': int(missing_before),
            'missing_after': int(missing_after),
            'improvement': f"{(missing_before - missing_after) / max(missing_before, 1) * 100:.1f}%"
        }
        
        # 2. Duplicates
        duplicates_before = raw_df.duplicated().sum()
        duplicates_after = clean_df.duplicated().sum()
        
        file_quality['duplicates'] = {
            'duplicates_before': int(duplicates_before),
            'duplicates_after': int(duplicates_after),
            'removed': int(duplicates_before - duplicates_after)
        }
        
        # 3. Validation violations
        viol_count = len(violations.get(file_name, pd.DataFrame()))
        total_rows = len(raw_df)
        
        file_quality['validation'] = {
            'violations': viol_count,
            'violation_rate': f"{(viol_count / total_rows * 100):.1f}%"
        }
        
        # 4. Row count
        file_quality['row_count'] = {
            'initial': len(raw_df),
            'final': len(clean_df),
            'removed': len(raw_df) - len(clean_df)
        }
        
        # 5. Score (0-100)
        completeness_score = 100 - (missing_before / max(total_rows * len(raw_df.columns), 1) * 100)
        duplicate_score = 100 - (duplicates_before / max(total_rows, 1) * 100)
        validation_score = 100 - (viol_count / max(total_rows, 1) * 100)
        
        file_quality['score'] = round((completeness_score + duplicate_score + validation_score) / 3, 1)
        
        quality_summary['file_quality'][file_name] = file_quality
        total_score += file_quality['score']
        file_count += 1
        
        # Recommendations
        if viol_count > total_rows * 0.1:
            quality_summary['recommendations'].append(
                f"{file_name}: High violation rate ({viol_count}/{total_rows}). Review data quality."
            )
        
        if missing_before > total_rows * 0.2:
            quality_summary['recommendations'].append(
                f"{file_name}: High missing value rate. Investigate data collection."
            )
    
    # Overall score
    quality_summary['overall_quality_score'] = round(total_score / max(file_count, 1), 1)
    
    # Overall recommendations
    if quality_summary['overall_quality_score'] < 70:
        quality_summary['recommendations'].append(
            "Overall data quality is below 70%. Consider improving data collection and cleaning processes."
        )
    elif quality_summary['overall_quality_score'] < 85:
        quality_summary['recommendations'].append(
            "Data quality is acceptable but can be improved. Focus on validation rules."
        )
    else:
        quality_summary['recommendations'].append(
            "Data quality is good. Data is ready for analysis and modeling."
        )
    
    return quality_summary

In [22]:
# import seaborn as sns

# تنظیم استایل
plt.style.use('seaborn-v0_8-darkgrid')
# sns.set_palette("husl")

def generate_analytical_outputs(processed_dir: Path) -> None:
    """
    Generate practical analytical figures based on available columns
    """
    figures_dir = processed_dir / 'figures'
    figures_dir.mkdir(parents=True, exist_ok=True)
    
    # Load integrated data
    integrated_path = processed_dir / 'integrated' / 'master_integrated_data.csv'
    if not integrated_path.exists():
        print("❌ Integrated data not found")
        return
    
    df = pd.read_csv(integrated_path)
    print(f"✅ Loaded {len(df)} rows")
    print(f"Columns: {df.columns.tolist()}\n")
    
    # ==========================================
    # Figure 1: Inventory Distribution by Warehouse (Bar Chart)
    # ==========================================
    print("📊 Creating Figure 1: Inventory by Warehouse...")
    try:
        fig1, ax1 = plt.subplots(figsize=(12, 6))
        
        if 'warehouse_id' in df.columns and 'on_hand_quantity' in df.columns:
            # گروه‌بندی موجودی بر اساس انبار
            wh_inv = df.groupby('warehouse_id')['on_hand_quantity'].sum().sort_values(ascending=False)
            
            # بار chart
            bars = ax1.bar(wh_inv.index, wh_inv.values, color='steelblue', edgecolor='black')
            ax1.set_title('Total Inventory by Warehouse', fontsize=16, fontweight='bold')
            ax1.set_xlabel('Warehouse', fontsize=12)
            ax1.set_ylabel('Total Inventory (units)', fontsize=12)
            ax1.grid(axis='y', alpha=0.3)
            
            # اضافه کردن مقدار روی هر ستون
            for bar, val in zip(bars, wh_inv.values):
                ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                        f'{val:,.0f}', ha='center', va='bottom', fontsize=10)
            
            plt.xticks(rotation=45)
            plt.tight_layout()
            fig1.savefig(figures_dir / 'inventory_by_warehouse.png', dpi=150, bbox_inches='tight')
            print("  ✅ Figure 1 saved: inventory_by_warehouse.png")
        else:
            print("  ⚠️  Required columns not found")
        plt.close()
    except Exception as e:
        print(f"  ❌ Figure 1 failed: {e}")
        plt.close()
    
    # ==========================================
    # Figure 2: Top 10 Products by Sales (Bar Chart)
    # ==========================================
    print("📊 Creating Figure 2: Top Products by Sales...")
    try:
        fig2, ax2 = plt.subplots(figsize=(12, 6))
        
        # پیدا کردن ستون فروش
        sales_col = None
        for col in df.columns:
            if 'sales' in col.lower() and ('30' in col or 'quantity' in col):
                sales_col = col
                break
        
        if sales_col and 'sku' in df.columns:
            # گروه‌بندی فروش بر اساس SKU
            sku_sales = df.groupby('sku')[sales_col].sum().sort_values(ascending=False)
            top_10 = sku_sales.head(10)
            
            # بار chart افقی
            bars = ax2.barh(top_10.index, top_10.values, color='coral', edgecolor='black')
            ax2.set_title('Top 10 Products by Sales', fontsize=16, fontweight='bold')
            ax2.set_xlabel('Total Sales (units)', fontsize=12)
            ax2.set_ylabel('SKU', fontsize=12)
            ax2.grid(axis='x', alpha=0.3)
            
            # اضافه کردن مقدار
            for bar, val in zip(bars, top_10.values):
                ax2.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                        f'{val:,.0f}', ha='left', va='center', fontsize=10)
            
            plt.tight_layout()
            fig2.savefig(figures_dir / 'top_products_by_sales.png', dpi=150, bbox_inches='tight')
            print(f"  ✅ Figure 2 saved: top_products_by_sales.png (using column: {sales_col})")
        else:
            print(f"  ⚠️  Sales column not found")
        plt.close()
    except Exception as e:
        print(f"  ❌ Figure 2 failed: {e}")
        plt.close()
    
    # ==========================================
    # Figure 3: Sales vs Inventory Scatter Plot
    # ==========================================
    print("📊 Creating Figure 3: Sales vs Inventory...")
    try:
        fig3, ax3 = plt.subplots(figsize=(10, 8))
        
        # پیدا کردن ستون‌ها
        sales_col = None
        inv_col = 'on_hand_quantity' if 'on_hand_quantity' in df.columns else None
        
        for col in df.columns:
            if 'sales' in col.lower() and ('30' in col or 'quantity' in col):
                sales_col = col
                break
        
        if sales_col and inv_col:
            # اسکاتر پلات
            scatter = ax3.scatter(df[sales_col], df[inv_col], 
                                alpha=0.6, c=df[sales_col], cmap='viridis', 
                                edgecolors='black', linewidth=0.5)
            
            ax3.set_title('Sales vs Inventory Relationship', fontsize=16, fontweight='bold')
            ax3.set_xlabel(f'Sales ({sales_col})', fontsize=12)
            ax3.set_ylabel('On-Hand Inventory', fontsize=12)
            ax3.grid(alpha=0.3)
            
            # اضافه کردن colorbar
            cbar = plt.colorbar(scatter, ax=ax3)
            cbar.set_label('Sales Volume', fontsize=10)
            
            # اضافه کردن خط میانه
            if df[inv_col].quantile(0.5) > 0:
                ax3.axhline(y=df[inv_col].median(), color='red', linestyle='--', 
                           alpha=0.5, label=f'Median Inventory: {df[inv_col].median():.0f}')
                ax3.legend()
            
            plt.tight_layout()
            fig3.savefig(figures_dir / 'sales_vs_inventory.png', dpi=150, bbox_inches='tight')
            print(f"  ✅ Figure 3 saved: sales_vs_inventory.png")
        else:
            print(f"  ⚠️  Required columns not found (sales_col={sales_col}, inv_col={inv_col})")
        plt.close()
    except Exception as e:
        print(f"  ❌ Figure 3 failed: {e}")
        plt.close()
    
    # ==========================================
    # Figure 4: Inventory Health Pie Chart
    # ==========================================
    print("📊 Creating Figure 4: Inventory Health...")
    try:
        fig4, ax4 = plt.subplots(figsize=(8, 8))
        
        # محاسبه وضعیت موجودی
        inv_col = 'on_hand_quantity' if 'on_hand_quantity' in df.columns else None
        sales_col = None
        for col in df.columns:
            if 'sales' in col.lower() and ('30' in col or 'quantity' in col):
                sales_col = col
                break
        
        if inv_col:
            # دسته‌بندی موجودی
            conditions = [
                (df[inv_col] == 0),  # Out of Stock
                (df[inv_col] < 50),  # Low Stock
                (df[inv_col] >= 50) & (df[inv_col] < 500),  # Medium Stock
                (df[inv_col] >= 500)  # High Stock
            ]
            labels = ['Out of Stock', 'Low Stock (<50)', 'Medium Stock (50-500)', 'High Stock (500+)']
            df['stock_category'] = pd.cut(df[inv_col], bins=[-1, 0, 50, 500, float('inf')], 
                                         labels=['Out of Stock', 'Low', 'Medium', 'High'])
            
            counts = df['stock_category'].value_counts()
            colors = ['#ff6b6b', '#ffd93d', '#6bcb77', '#4dabf7']
            
            # پای چارت
            wedges, texts, autotexts = ax4.pie(
                counts.values, 
                labels=counts.index, 
                autopct='%1.1f%%',
                colors=colors[:len(counts)],
                startangle=90,
                explode=[0.05] * len(counts)
            )
            
            ax4.set_title('Inventory Health Distribution', fontsize=16, fontweight='bold')
            
            # اضافه کردن تعداد
            for i, (label, count) in enumerate(counts.items()):
                texts[i].set_text(f'{label}\n({count:,})')
            
            plt.tight_layout()
            fig4.savefig(figures_dir / 'inventory_health.png', dpi=150, bbox_inches='tight')
            print(f"  ✅ Figure 4 saved: inventory_health.png")
        else:
            print(f"  ⚠️  Inventory column not found")
        plt.close()
    except Exception as e:
        print(f"  ❌ Figure 4 failed: {e}")
        plt.close()
    
    # ==========================================
    # Figure 5: Top Warehouses by SKU Count
    # ==========================================
    print("📊 Creating Figure 5: SKU Count by Warehouse...")
    try:
        fig5, ax5 = plt.subplots(figsize=(10, 6))
        
        if 'warehouse_id' in df.columns and 'sku' in df.columns:
            # تعداد SKU در هر انبار
            wh_sku_count = df.groupby('warehouse_id')['sku'].nunique().sort_values(ascending=False)
            
            # بار chart
            colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(wh_sku_count)))
            bars = ax5.bar(wh_sku_count.index, wh_sku_count.values, color=colors, edgecolor='black')
            
            ax5.set_title('Number of Unique SKUs by Warehouse', fontsize=16, fontweight='bold')
            ax5.set_xlabel('Warehouse', fontsize=12)
            ax5.set_ylabel('Number of SKUs', fontsize=12)
            ax5.grid(axis='y', alpha=0.3)
            
            # اضافه کردن مقدار
            for bar, val in zip(bars, wh_sku_count.values):
                ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                        f'{val}', ha='center', va='bottom', fontsize=10)
            
            plt.xticks(rotation=45)
            plt.tight_layout()
            fig5.savefig(figures_dir / 'sku_count_by_warehouse.png', dpi=150, bbox_inches='tight')
            print(f"  ✅ Figure 5 saved: sku_count_by_warehouse.png")
        else:
            print(f"  ⚠️  Required columns not found")
        plt.close()
    except Exception as e:
        print(f"  ❌ Figure 5 failed: {e}")
        plt.close()
    
    print("\n" + "="*50)
    print(f"✅ All figures saved to: {figures_dir}")
    print("="*50)

In [23]:
def run_etl_pipeline_with_audit():
    """
    Main ETL pipeline execution with comprehensive audit logging (Q8)
    """
    # Initialize audit logger
    audit = AuditLogger(LOG_DIR, "FMCG_ETL_Pipeline")
    audit.start()
    
    processed_data = {}
    summary = {
        'execution_time': datetime.datetime.now().isoformat(),
        'files_processed': [],
        'total_rows_processed': 0,
        'validation_summary': {},
        'cleaning_summary': {},
        'integration_summary': {},
        'analytics_summary': {}
    }
    
    try:
        # ==========================================
        # STEP 1: Load raw files
        # ==========================================
        audit.log_stage_start("EXTRACT", "Loading raw files from data/raw/")
        
        raw_data = load_raw_files(RAW_DIR)
        
        if not raw_data:
            audit.log_error(ValueError("No raw files loaded"), "EXTRACT stage")
            audit.finish("FAILED")
            return None, None, audit
        
        # Log files read
        file_paths = {name: RAW_DIR / f"{name}.csv" for name in raw_data.keys()}
        audit.log_files_read(file_paths)
        
        audit.log_rows_processed("EXTRACT", {name: len(df) for name, df in raw_data.items()})
        summary['total_rows_processed'] = sum(len(df) for df in raw_data.values())
        
        audit.log_stage_end("EXTRACT", "SUCCESS", {
            'files_loaded': len(raw_data),
            'total_rows': summary['total_rows_processed']
        })
        
        # ==========================================
        # STEP 2: Validate raw data
        # ==========================================
        audit.log_stage_start("VALIDATE", "Validating data against business rules (Q2)")
        
        warehouses = raw_data.get('warehouses')
        if warehouses is None:
            error = ValueError("Warehouses file not found!")
            audit.log_error(error, "VALIDATE stage")
            audit.finish("FAILED")
            return None, None, audit
        
        violations = validate_all_files(
            raw_data=raw_data,
            warehouses=warehouses,
            master_sku=None,
            allowed_status=None
        )
        
        audit.log_validation_results(violations)
        
        validation_summary = generate_validation_report(
            violations=violations,
            output_path=REPORT_DIR
        )
        
        summary['validation_summary'] = {
            file: {
                'total_violations': len(viol_df),
                'rules': viol_df['rule_id'].value_counts().to_dict() if not viol_df.empty else {}
            }
            for file, viol_df in violations.items()
        }
        
        # Get detailed row count info for audit
        clean_rows = {}
        for name, df in raw_data.items():
            if name in violations:
                clean_rows[name] = len(df) - len(violations[name])
            else:
                clean_rows[name] = len(df)
        
        audit.log_stage_end("VALIDATE", "SUCCESS", {
            'total_violations': sum(len(v) for v in violations.values()),
            'clean_rows': clean_rows
        })
        
        # ==========================================
        # STEP 3: Clean each file
        # ==========================================
        audit.log_stage_start("TRANSFORM", "Cleaning and standardizing data (Q3)")
        
        cleaning_functions = {
            'sales': clean_sales,
            'orders': clean_orders,
            'inventory': clean_inventory,
            'warehouses': clean_warehouses
        }
        
        cleaning_results = {}
        for file_name, df in raw_data.items():
            if file_name in cleaning_functions:
                initial_rows = len(df)
                cleaned_df = cleaning_functions[file_name](df)
                processed_data[file_name] = cleaned_df
                final_rows = len(cleaned_df)
                
                cleaning_results[file_name] = {
                    'initial_rows': initial_rows,
                    'final_rows': final_rows,
                    'removed_rows': initial_rows - final_rows,
                    'columns': len(cleaned_df.columns),
                    'removed_percent': ((initial_rows - final_rows) / initial_rows * 100) if initial_rows > 0 else 0
                }
                
                summary['cleaning_summary'][file_name] = cleaning_results[file_name]
            else:
                logger.warning(f"No cleaning function for {file_name}, keeping raw")
                processed_data[file_name] = df
        
        # Log cleaning results
        audit.log_cleaning_results(cleaning_results)
        
        audit.log_stage_end("TRANSFORM", "SUCCESS", {
            'files_cleaned': len(processed_data),
            'total_removed': sum(r['removed_rows'] for r in cleaning_results.values())
        })
        
        # ==========================================
        # STEP 4: Save processed files
        # ==========================================
        audit.log_stage_start("SAVE", "Saving cleaned files to data/processed/")
        
        saved_files = []
        for name, df in processed_data.items():
            output_path = PROCESSED_DIR / f"{name}_cleaned.csv"
            df.to_csv(output_path, index=False)
            saved_files.append(str(output_path))
        
        audit.log_stage_end("SAVE", "SUCCESS", {
            'files_saved': len(saved_files),
            'output_directory': str(PROCESSED_DIR)
        })
        
        # ==========================================
        # STEP 5: Integrate data (Q4)
        # ==========================================
        audit.log_stage_start("INTEGRATE", "Integrating data sources (Q4)")
        
        integrated_df, integration_summary = run_integration_pipeline(PROCESSED_DIR)
        
        if integrated_df is None:
            audit.log_error(ValueError("Integration failed!"), "INTEGRATE stage")
            audit.finish("FAILED")
            return processed_data, summary, audit
        
        summary['integration_summary'] = integration_summary
        audit.log_integration_results(integration_summary)
        
        audit.log_stage_end("INTEGRATE", "SUCCESS", {
            'final_rows': len(integrated_df),
            'final_columns': len(integrated_df.columns)
        })
        
        # ==========================================
        # STEP 6: Generate attention alerts (Q6)
        # ==========================================
        audit.log_stage_start("ANALYTICS_Q6", "Generating attention alerts (Q6)")
        
        q6_results = run_q6_analysis(PROCESSED_DIR)
        
        if q6_results:
            total_alerts = sum(len(df) for df in q6_results.values() if df is not None)
            summary['analytics_summary']['q6'] = {
                'total_alerts': total_alerts,
                'alert_types': {k: len(v) for k, v in q6_results.items() if v is not None}
            }
            
            audit.log_analytics_results({
                'q6_alerts_total': total_alerts,
                'alert_breakdown': {k: len(v) for k, v in q6_results.items() if v is not None}
            })
            
            audit.log_stage_end("ANALYTICS_Q6", "SUCCESS", {
                'total_alerts': total_alerts
            })
        
        # ==========================================
        # STEP 7: Verify reproducibility (Q7)
        # ==========================================
        audit.log_stage_start("REPRODUCIBILITY_Q7", "Verifying reproducibility (Q7)")
        
        q7_result = verify_reproducibility()
        summary['q7_reproducible'] = q7_result
        
        audit.log_stage_end("REPRODUCIBILITY_Q7", "SUCCESS" if q7_result else "WARNING", {
            'reproducible': q7_result
        })
        
        # ==========================================
        # STEP 8: Database load status (SQLite)
        # ==========================================
        audit.log_stage_start("DB_LOAD", "Loading data to SQLite database")
        
        db_path = BASE_DIR / 'database' / 'inventory.db'
        db_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Load data to SQLite
        import sqlite3
        conn = sqlite3.connect(db_path)
        
        tables_loaded = {}
        for name, df in processed_data.items():
            table_name = name
            df.to_sql(table_name, conn, if_exists='replace', index=False)
            tables_loaded[table_name] = len(df)
            audit.audit_logger.info(f"  ✅ Loaded {table_name}: {len(df)} rows")
        
        # Also load integrated data
        if integrated_df is not None:
            integrated_df.to_sql('integrated_data', conn, if_exists='replace', index=False)
            tables_loaded['integrated_data'] = len(integrated_df)
            audit.audit_logger.info(f"  ✅ Loaded integrated_data: {len(integrated_df)} rows")
        
        conn.close()
        
        audit.log_db_load_status(db_path, tables_loaded)
        
        audit.log_stage_end("DB_LOAD", "SUCCESS", {
            'db_path': str(db_path),
            'tables_loaded': len(tables_loaded),
            'total_rows_db': sum(tables_loaded.values())
        })
        
        # ==========================================
        # Generate final summary
        # ==========================================
        summary['total_files_processed'] = len(processed_data)
        summary['total_rows_final'] = sum(len(df) for df in processed_data.values())
        
        # Save summary as JSON
        summary_path = PROCESSED_DIR / 'processing_summary.json'
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=2, default=str)
        
        # STEP: Generate SQLite Database
        audit.log_stage_start("DB_CREATE", "Creating SQLite database (.db)")
        
        db_path = BASE_DIR / 'database' / 'inventory.db'
        tables_loaded = create_compressed_database(PROCESSED_DIR, db_path)
        
        audit.log_db_load_status(db_path, tables_loaded)
        audit.log_stage_end("DB_CREATE", "SUCCESS", {
            'db_path': str(db_path),
            'tables_loaded': len(tables_loaded)
        })
        
        # STEP: Generate Data Quality Summary
        audit.log_stage_start("QUALITY_SUMMARY", "Generating data quality summary")
        
        quality_summary = generate_data_quality_summary(raw_data, processed_data, violations, summary)
        
        # Save quality summary
        quality_path = PROCESSED_DIR / 'data_quality_summary.json'
        with open(quality_path, 'w') as f:
            json.dump(quality_summary, f, indent=2, default=str)
        
        audit.log_stage_end("QUALITY_SUMMARY", "SUCCESS", {
            'overall_score': quality_summary['overall_quality_score'],
            'file_count': len(quality_summary['file_quality'])
        })
        
        # STEP: Generate Analytical Outputs/Figures
        audit.log_stage_start("ANALYTICAL_FIGURES", "Generating analytical figures")
        
        generate_analytical_outputs(PROCESSED_DIR)
        
        audit.log_stage_end("ANALYTICAL_FIGURES", "SUCCESS", {
            'figures_dir': str(PROCESSED_DIR / 'figures')
        })
        
        # Finish audit
        audit.finish("SUCCESS")
        
        # Print audit summary
        print("\n" + "=" * 70)
        print("AUDIT SUMMARY (Q8)")
        print("=" * 70)
        audit_summary = audit.get_summary()
        for key, value in audit_summary.items():
            print(f"  {key}: {value}")
        print("=" * 70)
        print(f"Audit log: {audit.audit_log_file}")
        print(f"Audit JSON: {LOG_DIR / f'audit_{audit.execution_id}.json'}")
        print("=" * 70)
        
        return processed_data, summary, audit
        
    except Exception as e:
        audit.log_error(e, "PIPELINE_EXECUTION")
        audit.finish("FAILED")
        logger.error(f"Pipeline failed with error: {str(e)}")
        logger.exception("Full traceback:")
        return processed_data, summary, audit

In [24]:
if __name__ == "__main__":
    # Check if raw files exist
    if not RAW_DIR.exists():
        logger.error(f"Raw data directory not found: {RAW_DIR}")
        logger.info("Please ensure raw files are placed in the data/raw/ folder")
        sys.exit(1)
    
    # Run the ETL pipeline with audit logging
    processed_data, summary, audit = run_etl_pipeline_with_audit()
    
    if processed_data:
        # Display a sample of cleaned data
        print("\n" + "=" * 60)
        print("SAMPLE OF CLEANED DATA")
        print("=" * 60)
        for name, df in processed_data.items():
            print(f"\n{name.upper()} DATA (first 3 rows):")
            print(df.head(3))
            print(f"Shape: {df.shape}")
            print(f"Columns: {df.columns.tolist()}")
        
        # Display integrated data sample
        integrated_path = INTEGRATED_DIR / 'master_integrated_data.csv'
        if integrated_path.exists():
            integrated_df = pd.read_csv(integrated_path)
            print("\n" + "=" * 60)
            print("INTEGRATED DATA SAMPLE (Q4)")
            print("=" * 60)
            print(f"\nShape: {integrated_df.shape}")
            print(f"Columns: {integrated_df.columns.tolist()}")
            print(f"\nFirst 3 rows:")
            print(integrated_df.head(3))
        
        print("\n" + "=" * 60)
        print("ETL Pipeline execution completed!")
        print(f"Check audit log: {LOG_DIR / f'audit_{audit.execution_id}.log'}")
        print(f"Check audit JSON: {LOG_DIR / f'audit_{audit.execution_id}.json'}")
        print(f"Processed files saved in: {PROCESSED_DIR}")
        print(f"Integrated data saved in: {INTEGRATED_DIR}")
        print("=" * 60)
    else:
        print("\nPipeline failed - check the audit logs for errors.")

08:56:19 - AUDIT - ======================================================================


INFO:audit_20260906_085619:======================================================================


08:56:19 - AUDIT - AUDIT LOG - FMCG_ETL_Pipeline


INFO:audit_20260906_085619:AUDIT LOG - FMCG_ETL_Pipeline


08:56:19 - AUDIT - ======================================================================


...

08:56:59 - AUDIT - ======================================================================


INFO:audit_20260906_085619:======================================================================


08:56:59 - AUDIT - Audit data saved to: C:\Users\WelCome\Documents\studybuild\Mahshid28\logs\audit_20260906_085619.json


INFO:audit_20260906_085619:Audit data saved to: C:\Users\WelCome\Documents\studybuild\Mahshid28\logs\audit_20260906_085619.json



AUDIT SUMMARY (Q8)
  execution_id: 20260906_085619
  status: SUCCESS
  start_time: 2026-09-06T08:56:19.674239
  duration_seconds: 39.727466
  files_read: 4
  stages_completed: 10
  total_errors: 0
  total_warnings: 0
Audit log: C:\Users\WelCome\Documents\studybuild\Mahshid28\logs\audit_20260906_085619.log
Audit JSON: C:\Users\WelCome\Documents\studybuild\Mahshid28\logs\audit_20260906_085619.json

SAMPLE OF CLEANED DATA

SALES DATA (first 3 rows):
  transaction_id        date      sku warehouse_id  sales_quantity  revenue
0     TXN-001606  2026-06-05  sku0048        wh005              17  1098.95
1     TXN-000708  2026-08-16  sku0017        wh001               1  1226.31
2     TXN-001792  2026-08-08  sku0018        wh001              -2   580.43
Shape: (1800, 6)
Columns: ['transaction_id', 'date', 'sku', 'warehouse_id', 'sales_quantity', 'revenue']

ORDERS DATA (first 3 rows):
      order_id        date      sku warehouse_id  ordered_quantity  \
0  ORD-0000856  2026-07-31  sku0051     